# SCAM SLEUTH – Fake Job Posting Analysis

**Project:** Intelligent Fake Job and Internship Detection Using Data Mining  
**Dataset:** EMSCAD / Fake Job Postings dataset (`fake_job_postings.csv`)  

This notebook demonstrates data inspection, preprocessing, exploratory analysis, a text-classification baseline, clustering, and optional association-rule mining. The model output is a screening signal only; it does not prove that a job advertisement is fraudulent. Verify employers independently.

## How to run
1. Install the libraries listed in `requirements.txt`.
2. Place `fake_job_postings.csv` in the same folder as this notebook (or update `DATA_PATH`).
3. Run the cells from top to bottom.
4. Charts are saved in the `outputs/` folder.

The dataset contains 17,880 records in the supplied version. The `fraudulent` field is the target. The `in_balanced_dataset` field is metadata and is deliberately excluded from model features.

## 1. Import libraries and load the dataset

In [ ]:
from pathlib import Path
import re, html, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, accuracy_score,
                             precision_score, recall_score, f1_score)
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore")
DATA_PATH = Path("fake_job_postings.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("DataSet.csv")  # name used by some downloaded copies
if not DATA_PATH.exists():
    raise FileNotFoundError("Place fake_job_postings.csv or DataSet.csv beside this notebook.")

df = pd.read_csv(DATA_PATH)
print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()

## 2. Check data quality and target distribution

In [ ]:
# Normalise the target values: the supplied CSV may encode them as f/t.
if "fraudulent" not in df.columns:
    raise KeyError("Expected target column 'fraudulent' was not found.")

def to_binary(value):
    if pd.isna(value): return np.nan
    value = str(value).strip().lower()
    if value in {"t", "true", "1", "yes"}: return 1
    if value in {"f", "false", "0", "no"}: return 0
    return pd.to_numeric(value, errors="coerce")

df["fraudulent_label"] = df["fraudulent"].map(to_binary)
df = df.dropna(subset=["fraudulent_label"]).copy()
df["fraudulent_label"] = df["fraudulent_label"].astype(int)
print("Rows and columns:", df.shape)
print("Class counts:
", df["fraudulent_label"].value_counts().rename(index={0:"Legitimate",1:"Potentially fraudulent"}))
missing = (df.isna().mean()*100).sort_values(ascending=False)
missing.head(15)

In [ ]:
out_dir = Path("outputs")
out_dir.mkdir(exist_ok=True)
counts = df["fraudulent_label"].value_counts().sort_index()
plt.figure(figsize=(7,5))
plt.bar(["Legitimate", "Potentially fraudulent"], [counts.get(0,0), counts.get(1,0)])
plt.title("Job posting class distribution")
plt.ylabel("Number of postings")
plt.tight_layout(); plt.savefig(out_dir/"label_distribution.png", dpi=160); plt.show()

missing_plot = missing[missing > 0].head(12).sort_values()
plt.figure(figsize=(8,5)); plt.barh(missing_plot.index, missing_plot.values)
plt.title("Columns with the most missing values")
plt.xlabel("Missing values (%)")
plt.tight_layout(); plt.savefig(out_dir/"missing_values.png", dpi=160); plt.show()

## 3. Prepare text features

In [ ]:
# Combine only job-ad content fields. Do not use the target or metadata as predictors.
text_columns = [c for c in ["title", "company_profile", "description", "requirements", "benefits"] if c in df.columns]
if not text_columns:
    raise ValueError("No expected text columns were found in the dataset.")

def clean_text(value):
    if pd.isna(value): return ""
    value = html.unescape(str(value))
    value = re.sub(r"<[^>]+>", " ", value)  # remove HTML tags
    value = re.sub(r"https?://\S+|www\.\S+", " URL ", value)
    value = re.sub(r"[^A-Za-z0-9$%]+", " ", value)
    return re.sub(r"\s+", " ", value).strip().lower()

df["combined_text"] = df[text_columns].fillna("").astype(str).agg(" ".join, axis=1).map(clean_text)
X_train, X_test, y_train, y_test = train_test_split(
    df["combined_text"], df["fraudulent_label"], test_size=0.20,
    random_state=42, stratify=df["fraudulent_label"])
print("Training rows:", len(X_train), "Test rows:", len(X_test))

## 4. Train and evaluate a TF-IDF + Logistic Regression baseline

In [ ]:
# Vectorisation is inside the Pipeline, so vocabulary is learned from training data only.
model = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=50000, ngram_range=(1,2),
                               min_df=2, stop_words="english", sublinear_tf=True)),
    ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced",
                                       random_state=42))
])
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
metrics = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred, zero_division=0),
    "Recall": recall_score(y_test, y_pred, zero_division=0),
    "F1-score": f1_score(y_test, y_pred, zero_division=0)
}
print(pd.Series(metrics).round(4))
print("\nClassification report:\n", classification_report(y_test, y_pred, target_names=["Legitimate", "Potentially fraudulent"], zero_division=0))

ConfusionMatrixDisplay.from_predictions(y_test, y_pred,
    display_labels=["Legitimate", "Potentially fraudulent"], cmap="Blues", values_format="d")
plt.title("Confusion matrix – held-out test set")
plt.tight_layout(); plt.savefig(out_dir/"confusion_matrix.png", dpi=160); plt.show()
plt.figure(figsize=(7,4)); plt.bar(metrics.keys(), metrics.values())
plt.ylim(0,1); plt.ylabel("Score"); plt.title("Baseline model performance")
plt.tight_layout(); plt.savefig(out_dir/"model_metrics.png", dpi=160); plt.show()

## 5. Explore frequent terms in flagged examples

In [ ]:
# Descriptive term counts only; these terms are not proof of fraud.
from sklearn.feature_extraction.text import CountVectorizer
flagged_text = df.loc[df["fraudulent_label"] == 1, "combined_text"]
if len(flagged_text):
    cv = CountVectorizer(stop_words="english", max_features=20, ngram_range=(1,2), min_df=2)
    mat = cv.fit_transform(flagged_text)
    term_counts = np.asarray(mat.sum(axis=0)).ravel()
    top_terms = pd.Series(term_counts, index=cv.get_feature_names_out()).sort_values(ascending=False).head(15)
    display(top_terms.to_frame("count"))
    plt.figure(figsize=(9,5)); top_terms.sort_values().plot(kind="barh")
    plt.title("Frequent terms in labelled fraudulent postings")
    plt.xlabel("Count in postings"); plt.tight_layout()
    plt.savefig(out_dir/"frequent_terms.png", dpi=160); plt.show()
else:
    print("No positive-label rows available for term analysis.")

## 6. K-Means clustering of text postings

In [ ]:
# Unsupervised grouping; clusters are not fraud labels.
cluster_vectorizer = TfidfVectorizer(max_features=12000, min_df=2, stop_words="english")
X_cluster = cluster_vectorizer.fit_transform(df["combined_text"])
kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
df["text_cluster"] = kmeans.fit_predict(X_cluster)
svd = TruncatedSVD(n_components=2, random_state=42)
coords = svd.fit_transform(X_cluster)
plt.figure(figsize=(8,6))
plt.scatter(coords[:,0], coords[:,1], c=df["text_cluster"], s=8, alpha=.45)
plt.title("K-Means clusters (2D SVD projection)")
plt.xlabel("SVD component 1"); plt.ylabel("SVD component 2")
plt.tight_layout(); plt.savefig(out_dir/"cluster_visualisation.png", dpi=160); plt.show()
print(pd.crosstab(df["text_cluster"], df["fraudulent_label"],
                  rownames=["Cluster"], colnames=["Actual label"]))

## 7. Optional association-rule mining (Apriori)

In [ ]:
# Optional: install mlxtend if needed. Uses simple posting attributes as transactions.
# Run: %pip install mlxtend
try:
    from mlxtend.frequent_patterns import apriori, association_rules
    from mlxtend.preprocessing import TransactionEncoder
    categorical = [c for c in ["employment_type", "required_experience", "required_education", "industry", "function"] if c in df.columns]
    sample = df[categorical].fillna("Unknown").astype(str).head(5000)
    transactions = [[f"{col}={row[col]}" for col in categorical if row[col] != "Unknown"] for _, row in sample.iterrows()]
    te = TransactionEncoder()
    basket = pd.DataFrame(te.fit(transactions).transform(transactions), columns=te.columns_)
    itemsets = apriori(basket, min_support=0.02, use_colnames=True, max_len=3)
    if len(itemsets):
        rules = association_rules(itemsets, metric="confidence", min_threshold=0.5)
        display(rules.sort_values(["lift", "confidence"], ascending=False).head(10))
    else:
        print("No frequent itemsets met the selected support threshold. Try a lower threshold carefully.")
except ImportError:
    print("Optional package mlxtend is not installed. Install it with: %pip install mlxtend")

## 8. Save a compact results summary

In [ ]:
summary = pd.DataFrame([{
    "dataset_rows": len(df),
    "legitimate_postings": int((df["fraudulent_label"] == 0).sum()),
    "labelled_fraudulent_postings": int((df["fraudulent_label"] == 1).sum()),
    **{k.lower().replace("-", "_"): float(v) for k,v in metrics.items()}
}])
summary.to_csv(out_dir/"model_summary.csv", index=False)
summary

## 9. Interpretation and limitations
- The classifier identifies patterns associated with the dataset labels; it cannot establish that a real-world employer is fraudulent.
- The dataset is imbalanced, so precision, recall, F1-score and the confusion matrix should be reviewed alongside accuracy.
- Results can vary with the train/test split, preprocessing choices and library versions.
- Clusters are unsupervised text groupings and must not be interpreted as verified scam categories.
- Never share personal information or pay fees solely because an advertisement appears in a dataset or receives a model prediction. Verify the employer through independent, trusted channels.